# Causal SQIL on HumanoidMaze Medium (v2)


In [1]:
import random
import copy
import torch
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import HumanoidMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *
from causal_rl.algo.imitation.gail.core_net import ContinuousActor
from causal_rl.algo.imitation.gail.causal_gail import *
from causal_rl.algo.imitation.sqil.core_net import SQILQNetwork
from causal_rl.algo.imitation.sqil.causal_sqil import (
    SQILReplayBuffer, initialize_expert_buffer,
    rollout_sqil_episode, sac_update, soft_update,
    evaluate_sqil_policy,
)

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
/home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '4'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
num_steps = 2000
seed = 0
lookback = 2
hidden_dims = {'C'}

random.seed(seed)
torch.manual_seed(seed)

lookback, hidden_dims

(2, {'C'})

In [4]:
# for training: regular W, C hidden
train_env = HumanoidMazePCH(env_id='humanoidmaze-large-navigate-singletask-task1-v0', num_steps=num_steps, expert_mode=True, custom_hidden=hidden_dims, seed=seed, success_radius=20.0)

# for eval: corrupted W, C hidden
eval_env = HumanoidMazePCH(env_id='humanoidmaze-large-navigate-singletask-task1-v0', num_steps=num_steps, expert_mode=False, seed=seed, success_radius=20.0)

## Causal Graph Analysis

In [5]:
# to save time; conceptually the same
small_steps = lookback + 1
small_env = HumanoidMazePCH(num_steps=small_steps, seed=seed)
G = parse_graph(small_env.get_graph)
X_small = {f'X{t}' for t in range(small_steps)}
Y = f'Y{small_steps}'

X = {f'X{t}' for t in range(num_steps)}
obs_prefix = train_env.env.observed_unobserved_vars[0]

In [6]:
Z_sets = find_sequential_pi_backdoor(G, X_small, Y, obs_prefix)

base_step = small_steps - 1
base_Z_set = Z_sets[f'X{base_step}']

for i in range(base_step + 1, num_steps):
    updated_base_Z_set = set()
    for v in base_Z_set:
        updated_base_Z_set.add(f'{v[0]}{int(v[1:]) + i - lookback}')

    Z_sets[f'X{i}'] = updated_base_Z_set

Z_sets['X1']

{'A0', 'A1', 'E0', 'E1', 'H0', 'H1', 'J0', 'J1', 'P0', 'P1', 'V0', 'V1', 'X0'}

## Expert Trajectories

In [7]:
# for eval: corrupted W, O shown
expert_traj_env = HumanoidMazePCH(num_steps=num_steps, expert_mode=True)
# load model
MODEL_PATH = '/home/et2842/causal/causalrl/models/humanoidmaze_medium_expert_finetuned.pt'
expert_ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)

expert_action_bounds = (expert_ckpt['action_bounds_low'], expert_ckpt['action_bounds_high'])

expert_model = ContinuousPolicyNN(
    input_dim=expert_ckpt['input_dim'],
    action_dim=expert_ckpt['num_actions'],
    hidden_dim=256,
    num_blocks=expert_ckpt['num_blocks'],
    dropout=expert_ckpt['dropout'],
    layernorm=expert_ckpt['layernorm'],
    final_tanh=expert_ckpt['final_tanh'],
    action_bounds=expert_action_bounds,
).to(device)

expert_model.load_state_dict(expert_ckpt['state_dict'])
expert_model.eval()

expert_slots = expert_ckpt['slots']
expert_Z_trim = expert_ckpt['Z_trim']
expert_dims = expert_ckpt['dims']
expert_lookback = expert_ckpt['lookback']

expert_policy = shared_policy_fn_long_horizon(expert_model, expert_slots, expert_Z_trim, continuous=True, device=device)
expert_policies = make_shared_policy_dict(expert_policy)
expert_num_eval_eps = 500

records = collect_imitator_trajectories(
    env=expert_traj_env,
    policies=expert_policies,
    num_episodes=expert_num_eval_eps,
    max_steps=num_steps,
    show_progress=True
)

len(records)

Starting episode 1/500...


  Episode 1 ended at step 2000 (terminated: False, truncated: True).
Starting episode 2/500...


  Episode 2 ended at step 2000 (terminated: False, truncated: True).
Starting episode 3/500...


  Episode 3 ended at step 2000 (terminated: False, truncated: True).
Starting episode 4/500...


  Episode 4 ended at step 684 (terminated: True, truncated: False).
Starting episode 5/500...


...

  Episode 497 ended at step 2000 (terminated: False, truncated: True).
Starting episode 498/500...


  Episode 498 ended at step 2000 (terminated: False, truncated: True).
Starting episode 499/500...


  Episode 499 ended at step 2000 (terminated: False, truncated: True).
Starting episode 500/500...


  Episode 500 ended at step 2000 (terminated: False, truncated: True).
Finished collecting imitator trajectories.


817519

In [8]:
dims = {
    'P': 2,
    'A': 21,
    'H': 1,
    'E': 12,
    'V': 3,
    # 'C': 3,
    'J': 27,
    'W': 2,
    'X': 21
}

In [9]:
sample_obs = records[0]['obs']

# Trim Z-sets to the lookback window
causal_Z_trim = trim_Z_sets(Z_sets, lookback=lookback)

# Build windowed encoders that depend on relative lags
causal_encode, causal_z_dim, causal_slots = build_windowed_z_encoder(
    causal_Z_trim,
    dims=dims,
    lookback=lookback,
)

encode = causal_encode
z_dim = causal_z_dim
Z_trim = causal_Z_trim
causal_z_dim

240

## Hyperparameters

In [10]:
# Shared SAC hyperparameters
total_timesteps = 4_000_000
batch_size = 256
gamma = 0.99
tau = 0.005
actor_lr = 3e-4
critic_lr = 3e-4
alpha_lr = 3e-4
hidden_dim = 256
buffer_capacity = 1_000_000
expert_capacity_ratio = 0.5
start_steps = 5_000
log_every = 50
eval_episodes = 10
max_grad_norm = 1.0
utd_ratio = 0.5  # update-to-data ratio: 1 gradient update per 4 env steps

# Actor architecture (match GAIL)
num_blocks_actor = 3
dropout_actor = 0.05
layernorm_actor = True

# Environment action space
action_dim = train_env.env.action_space.shape[0]
action_low = float(train_env.env.action_space.low.min())
action_high = float(train_env.env.action_space.high.max())
target_entropy = -float(action_dim)

## Network Initialization

In [11]:
actor = ContinuousActor(
    num_inputs=z_dim, num_outputs=action_dim,
    hidden_size=hidden_dim, std=0.0,
    action_low=action_low, action_high=action_high,
    num_blocks=num_blocks_actor, dropout=dropout_actor, layernorm=layernorm_actor,
).to(device)

q1 = SQILQNetwork(z_dim, action_dim, hidden_dim,
                   num_blocks=num_blocks_actor, dropout=dropout_actor,
                   layernorm=layernorm_actor).to(device)
q2 = SQILQNetwork(z_dim, action_dim, hidden_dim,
                   num_blocks=num_blocks_actor, dropout=dropout_actor,
                   layernorm=layernorm_actor).to(device)
tq1 = copy.deepcopy(q1)
tq2 = copy.deepcopy(q2)
for p in tq1.parameters(): p.requires_grad = False
for p in tq2.parameters(): p.requires_grad = False

actor_optim = torch.optim.Adam(actor.parameters(), lr=actor_lr)
q1_optim = torch.optim.Adam(q1.parameters(), lr=critic_lr)
q2_optim = torch.optim.Adam(q2.parameters(), lr=critic_lr)

# Cosine LR schedule for critics
estimated_total_updates = int(total_timesteps * utd_ratio)
q1_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(q1_optim, T_max=estimated_total_updates)
q2_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(q2_optim, T_max=estimated_total_updates)

# Automatic entropy tuning
log_alpha = torch.zeros(1, requires_grad=True, device=device)
alpha_optim = torch.optim.Adam([log_alpha], lr=alpha_lr)

buffer = SQILReplayBuffer(buffer_capacity, expert_capacity_ratio)
initialize_expert_buffer(records, encode, buffer, device)

Expert buffer: 500000 transitions from 500 episodes


## Training

In [12]:
best_eval = -float('inf')
best_state_dict = copy.deepcopy(actor.state_dict())

ts = 0
ep = 0
logs = []

while ts < total_timesteps:
    ep_data = rollout_sqil_episode(
        train_env, actor, buffer, encode,
        num_steps, device, deterministic=False, seed=seed + 0 + ep
    )
    ts += ep_data['episode_length']
    ep += 1

    if ts > start_steps and len(buffer.policy_buffer) >= batch_size // 2:
        n_updates = max(1, int(ep_data['episode_length'] * utd_ratio))
        for _ in range(n_updates):
            sac_update(
                q1, q2, tq1, tq2, actor, log_alpha, target_entropy,
                q1_optim, q2_optim, actor_optim, alpha_optim,
                buffer, batch_size, gamma, device, max_grad_norm,
            )
            soft_update(q1, tq1, tau)
            soft_update(q2, tq2, tau)

            q1_scheduler.step()
            q2_scheduler.step()

            # Alpha clamping (stability fix, matches IQ-Learn)
            with torch.no_grad():
                log_alpha.clamp_(min=np.log(0.001), max=np.log(0.1))

    if ep % log_every == 0:
        eval_ret = evaluate_sqil_policy(
            train_env, actor, encode, num_steps, device, eval_episodes, seed=42
        )
        logs.append({
            'episode': ep, 'timesteps': ts,
            'eval_return': eval_ret, 'train_return': ep_data['episode_return'],
            'alpha': log_alpha.exp().item(),
        })
        print(
            f"[Causal SQIL ep {ep}] "
            f"ts={ts}, eval={eval_ret:.2f}, "
            f"train={ep_data['episode_return']:.2f}, "
            f"alpha={log_alpha.exp().item():.4f}"
        )

        # Best checkpoint tracking
        if eval_ret > best_eval:
            best_eval = eval_ret
            best_state_dict = copy.deepcopy(actor.state_dict())

# Restore best
actor.load_state_dict(best_state_dict)
print(f"Restored best checkpoint with eval={best_eval:.2f}")

[Causal SQIL ep 50] ts=100000, eval=-845.18, train=-1156.09, alpha=0.0069


[Causal SQIL ep 100] ts=200000, eval=-836.42, train=-549.14, alpha=0.0081


[Causal SQIL ep 150] ts=300000, eval=-809.88, train=-654.55, alpha=0.0074


[Causal SQIL ep 200] ts=400000, eval=-815.93, train=-854.09, alpha=0.0071


[Causal SQIL ep 250] ts=500000, eval=-793.25, train=-1331.62, alpha=0.0074


...

[Causal SQIL ep 1800] ts=3600000, eval=-806.52, train=-877.61, alpha=0.0076


[Causal SQIL ep 1850] ts=3700000, eval=-813.82, train=-923.97, alpha=0.0078


[Causal SQIL ep 1900] ts=3800000, eval=-818.41, train=-968.05, alpha=0.0076


[Causal SQIL ep 1950] ts=3900000, eval=-830.73, train=-726.45, alpha=0.0077


[Causal SQIL ep 2000] ts=4000000, eval=-810.29, train=-812.19, alpha=0.0079
Restored best checkpoint with eval=-793.25


In [13]:
causal_sqil_policy = make_gail_policy(actor, encode, device=device, deterministic=True)
causal_sqil_policies = make_shared_policy_dict(causal_sqil_policy)

## Save Model

In [14]:
SAVE_DIR = '/home/et2842/causal/causalrl/models'
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_PATH = os.path.join(SAVE_DIR, 'csqil_humlarge.pt')

ckpt = {
    'state_dict': actor.state_dict(),
    'z_dim': causal_z_dim,
    'action_dim': action_dim,
    'hidden_size_actor': hidden_dim,
    'num_blocks_actor': num_blocks_actor,
    'dropout_actor': dropout_actor,
    'layernorm_actor': layernorm_actor,
    'final_tanh': True,
    'action_bounds_low': eval_env.env.action_space.low,
    'action_bounds_high': eval_env.env.action_space.high,
    'Z_sets': causal_Z_trim,
    'dims': dims,
    'lookback': lookback,
}

torch.save(ckpt, MODEL_PATH)
print(f'Saved to: {MODEL_PATH}')

Saved to: /home/et2842/causal/causalrl/models/csqil_humlarge.pt


## Evaluation

In [15]:
num_eval_eps = 1000
csqil_returns = collect_imitator_trajectories(
    env=eval_env,
    policies=causal_sqil_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True,
    seed=seed + 90210,
)

len(csqil_returns)

Starting episode 1/1000...


  Episode 1 ended at step 2000 (terminated: False, truncated: True).
Starting episode 2/1000...


  Episode 2 ended at step 2000 (terminated: False, truncated: True).
Starting episode 3/1000...


  Episode 3 ended at step 2000 (terminated: False, truncated: True).
Starting episode 4/1000...


  Episode 4 ended at step 2000 (terminated: False, truncated: True).
Starting episode 5/1000...


...

  Episode 633 ended at step 2000 (terminated: False, truncated: True).
Starting episode 634/1000...


  Episode 634 ended at step 2000 (terminated: False, truncated: True).
Starting episode 635/1000...


  Episode 635 ended at step 2000 (terminated: False, truncated: True).
Starting episode 636/1000...


  Episode 636 ended at step 2000 (terminated: False, truncated: True).
Starting episode 637/1000...


  Episode 637 ended at step 2000 (terminated: False, truncated: True).
Starting episode 638/1000...


In [ ]:
csqil_episode_rewards = defaultdict(float)
for rec in csqil_returns:
    ep = rec['episode']
    csqil_episode_rewards[ep] += float(rec['reward'])

csqil_rewards = [csqil_episode_rewards[e] for e in range(num_eval_eps)]
sum(csqil_rewards) / num_eval_eps

In [ ]:
mean_reward = np.mean(csqil_rewards)
std_reward = np.std(csqil_rewards)

print(f"E[Y]          = {mean_reward:.4f}")
print(f"Std[Y]        = {std_reward:.4f}")
print(f"E[Y] ± Std[Y] = {mean_reward:.4f} ± {std_reward:.4f}")

In [ ]:
# success rate: % of episodes solved in under 1000 steps
ep_lengths = defaultdict(int)
for rec in csqil_returns:
    ep_lengths[rec['episode']] += 1

lengths = np.array([ep_lengths[e] for e in range(num_eval_eps)])
successes = lengths < num_steps
success_rate = successes.mean()
se = np.sqrt(success_rate * (1 - success_rate) / num_eval_eps)

print(f"Success rate   = {100 * success_rate:.2f}% ({successes.sum()}/{num_eval_eps} episodes)")
print(f"Std error      = {100 * se:.2f}%")

In [ ]:
# successful episode lengths
success_lengths = lengths[successes]

if len(success_lengths) > 0:
    print(f"Successful episode lengths (n={len(success_lengths)}):")
    print(f"  Mean   = {np.mean(success_lengths):.2f}")
    print(f"  Std    = {np.std(success_lengths):.2f}")
    print(f"  Median = {np.median(success_lengths):.0f}")
    print(f"  Min    = {np.min(success_lengths)}")
    print(f"  Max    = {np.max(success_lengths)}")
    print(f"  25th%  = {np.percentile(success_lengths, 25):.0f}")
    print(f"  75th%  = {np.percentile(success_lengths, 75):.0f}")
else:
    print("No episodes were solved.")